In [157]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aabdollahii/diabetes-ontl/Diabetes_large.owl


# Cell 1 — Imports


In [158]:
import os
import re
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from rdflib import Graph, URIRef, BNode, Literal
from rdflib.namespace import RDF, RDFS, OWL, DC, DCTERMS, SKOS

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)


# Cell 2 — Config


In [159]:
set_seed(42)

OWL_PATH = Path("/kaggle/input/datasets/aabdollahii/diabetes-ontl/Diabetes_large.owl")
WORK_DIR = Path("/kaggle/working/diabetes_ontology_mlm")
WORK_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

MAX_LENGTH = 128
MLM_PROBABILITY = 0.15
TRAIN_TEST_SPLIT = 0.1

NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 2
LOGGING_STEPS = 100
EVAL_STRATEGY = "epoch"
SAVE_STRATEGY = "epoch"

OUTPUT_DIR = WORK_DIR / "pubmedbert_diabetes_ontology_mlm"
CORPUS_TXT = WORK_DIR / "ontology_corpus.txt"
TRAIN_TXT = WORK_DIR / "train.txt"
VALID_TXT = WORK_DIR / "valid.txt"
SENTENCE_CSV = WORK_DIR / "ontology_sentences.csv"


# Cell 3 — Load ontology


In [160]:
if not OWL_PATH.exists():
    raise FileNotFoundError(f"Ontology file not found: {OWL_PATH}")

g = Graph()
g.parse(str(OWL_PATH))

print("Triples:", len(g))
print("Namespaces:", len(list(g.namespaces())))


Triples: 150335
Namespaces: 31


In [161]:
def local_name(uri):
    s = str(uri)
    if "#" in s:
        return s.rsplit("#", 1)[-1]
    return s.rstrip("/").rsplit("/", 1)[-1]

def safe_literal_text(x):
    text = str(x)
    text = text.replace("\n", " ").replace("\t", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def get_label(graph, node):
    for pred in [RDFS.label, SKOS.prefLabel, DC.title, DCTERMS.title]:
        val = graph.value(node, pred)
        if val:
            return safe_literal_text(val)
    return local_name(node)

def camel_to_words(text):
    text = str(text).replace("_", " ").replace("-", " ")
    text = re.sub(r"(?<!^)(?=[A-Z])", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_node_text(text):
    text = safe_literal_text(text)
    text = camel_to_words(text)
    text = re.sub(r"\(.*?SNOMED.*?\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\(.*?ICD.*?\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip(" .,:;")
    return text

def node_text(graph, node):
    if node is None:
        return ""
    
    if isinstance(node, Literal):
        return clean_node_text(str(node))
    
    text = get_label(graph, node)
    text = clean_node_text(text)
    
    if not text:
        text = clean_node_text(local_name(node))
    
    return text

def predicate_to_phrase(pred_text):
    pred_text = clean_node_text(pred_text)
    
    mapping = {
        "sub Class Of": "is a subclass of",
        "type": "is a",
        "has Age": "has age",
        "has B M I": "has BMI",
        "has Blood Pressure": "has blood pressure",
        "has Glucose": "has glucose",
        "has Outcome": "has outcome",
        "has Diabetes Pedigree Function": "has diabetes pedigree function",
        "leads To": "leads to",
        "causes": "causes",
        "associated With": "is associated with",
        "related To": "is related to",
        "has Risk Factor": "has risk factor",
        "has Symptom": "has symptom",
        "treated By": "is treated by"
    }
    
    return mapping.get(pred_text, pred_text.lower())

def is_noise_text(text):
    if not text:
        return True
    
    if text.lower() in {"thing", "class", "named individual"}:
        return True
    
    if len(text.strip()) == 0:
        return True
    
    return False


# Cell 5 — Extract ontology schema sets


In [162]:
classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
object_properties = set(g.subjects(RDF.type, OWL.ObjectProperty))
datatype_properties = set(g.subjects(RDF.type, OWL.DatatypeProperty))
annotation_properties = set(g.subjects(RDF.type, OWL.AnnotationProperty))
named_individuals = set(g.subjects(RDF.type, OWL.NamedIndividual))

print("Classes:", len(classes))
print("Object properties:", len(object_properties))
print("Datatype properties:", len(datatype_properties))
print("Annotation properties:", len(annotation_properties))
print("Named individuals:", len(named_individuals))


Classes: 65
Object properties: 21
Datatype properties: 5
Annotation properties: 1
Named individuals: 21558


# Cell 6 — Triple to sentence


In [163]:
def triple_to_sentence(graph, s, p, o):
    s_text = node_text(graph, s)
    p_text = node_text(graph, p)
    o_text = node_text(graph, o)

    if is_noise_text(s_text) or is_noise_text(p_text) or is_noise_text(o_text):
        return None

    if p == RDFS.subClassOf:
        return f"{s_text} is a subclass of {o_text}."
    
    if p == RDF.type:
        if o in {OWL.Class, RDFS.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.AnnotationProperty, OWL.NamedIndividual, OWL.Ontology}:
            return None
        return f"{s_text} is a {o_text}."
    
    phrase = predicate_to_phrase(p_text)
    return f"{s_text} {phrase} {o_text}."


# Cell 7 — Generate raw sentence corpus


In [164]:
records = []
seen = set()

for s, p, o in g:
    if not isinstance(p, URIRef):
        continue
    
    sentence = triple_to_sentence(g, s, p, o)
    if sentence is None:
        continue
    
    sentence = re.sub(r"\s+", " ", sentence).strip()
    if len(sentence) < 5:
        continue
    
    if sentence not in seen:
        seen.add(sentence)
        records.append({
            "subject": str(s),
            "predicate": str(p),
            "object": str(o),
            "sentence": sentence
        })

sent_df = pd.DataFrame(records)
print("Generated sentences:", len(sent_df))
sent_df.head(20)


Generated sentences: 128684


,subject,predicate,object,sentence
0,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 004046 is a Clinical Concept.
1,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 006005 leads to C Syntheti...
2,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 011248 leads to C Syntheti...
3,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 017177 leads to C Syntheti...
4,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 011460 is a Clinical Concept.
5,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 004145 leads to C Syntheti...
6,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 003555 is a Clinical Concept.
7,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.semanticweb.org/laptop-hp/ontologie...,560 is a Patient.
8,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 003121 leads to C Syntheti...
9,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,http://www.semanticweb.org/laptop-hp/ontologie...,C Synthetic Concept 019636 leads to C Syntheti...


In [165]:


def local_name(value):
    value = str(value)
    
    if "#" in value:
        value = value.rsplit("#", 1)[-1]
    
    value = value.rstrip("/").rsplit("/", 1)[-1]
    return value

def readable_name(value):
    name = local_name(value)
    name = name.replace("_", " ")
    name = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", name)
    name = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", name)
    return re.sub(r"\s+", " ", name).strip()

print("Triple count:", len(g))
print("Ontology resources:", len(set(g.subjects()) | set(g.predicates()) | set(g.objects())))


Triple count: 150335
Ontology resources: 21704


In [166]:
from collections import Counter
from rdflib import URIRef, BNode
from rdflib.namespace import RDF, RDFS, OWL
import pandas as pd

ignored_class_like_values = {
    RDF.Property,
    OWL.Class,
    OWL.ObjectProperty,
    OWL.DatatypeProperty,
    OWL.AnnotationProperty,
    OWL.NamedIndividual,
    OWL.Restriction,
}

def is_valid_class_uri(value):
    if not isinstance(value, URIRef):
        return False
    
    if value in ignored_class_like_values:
        return False
    
    text = str(value)
    
    if text.startswith(str(RDF)):
        return False
    
    if text.startswith(str(RDFS)):
        return False
    
    if text.startswith(str(OWL)):
        return False
    
    return True

class_uris = set()

for class_uri in g.subjects(RDF.type, OWL.Class):
    if is_valid_class_uri(class_uri):
        class_uris.add(class_uri)

for child, parent in g.subject_objects(RDFS.subClassOf):
    if is_valid_class_uri(child):
        class_uris.add(child)
    if is_valid_class_uri(parent):
        class_uris.add(parent)

for subject, class_uri in g.subject_objects(RDF.type):
    if is_valid_class_uri(class_uri):
        class_uris.add(class_uri)

for property_uri in list(g.subjects(RDF.type, OWL.ObjectProperty)) + list(g.subjects(RDF.type, OWL.DatatypeProperty)):
    for domain_uri in g.objects(property_uri, RDFS.domain):
        if is_valid_class_uri(domain_uri):
            class_uris.add(domain_uri)
    
    for range_uri in g.objects(property_uri, RDFS.range):
        if is_valid_class_uri(range_uri):
            class_uris.add(range_uri)

class_rows = []

for class_uri in sorted(class_uris, key=lambda value: readable_name(value)):
    instance_count = sum(
        1 for _ in g.subjects(RDF.type, class_uri)
    )
    
    direct_subclass_count = sum(
        1 for _ in g.subjects(RDFS.subClassOf, class_uri)
    )
    
    parent_classes = [
        readable_name(parent)
        for parent in g.objects(class_uri, RDFS.subClassOf)
        if is_valid_class_uri(parent)
    ]
    
    class_rows.append({
        "class_uri": str(class_uri),
        "class_name": readable_name(class_uri),
        "instance_count": instance_count,
        "direct_subclass_count": direct_subclass_count,
        "parent_classes": " | ".join(parent_classes)
    })

classes_df = pd.DataFrame(class_rows)

classes_df = classes_df.sort_values(
    ["class_name"],
    ascending=True
).reset_index(drop=True)

print("Detected classes:", len(classes_df))
display(classes_df)


Detected classes: 67


,class_uri,class_name,instance_count,direct_subclass_count,parent_classes
0,http://www.semanticweb.org/laptop-hp/ontologie...,Aerobic Exercise,1,1,Exercise Activity
1,http://www.semanticweb.org/laptop-hp/ontologie...,Age,0,0,
2,http://www.semanticweb.org/laptop-hp/ontologie...,BMI,0,4,
3,http://www.semanticweb.org/laptop-hp/ontologie...,Blood Pressure,0,4,
4,http://www.semanticweb.org/laptop-hp/ontologie...,Borderline Hb A1c,0,0,Hb A1c Level
...,...,...,...,...,...
62,http://www.semanticweb.org/laptop-hp/ontologie...,Walking Exercise,1,0,Aerobic Exercise
63,http://www.semanticweb.org/laptop-hp/ontologie...,Weight Loss Intervention,1,0,Lifestyle Factor
64,http://www.semanticweb.org/laptop-hp/ontologie...,Weight Management Status,0,0,Clinical Concept
65,http://www.w3.org/2001/XMLSchema#byte,byte,0,0,


In [167]:
subclass_rows = []

for child, parent in g.subject_objects(RDFS.subClassOf):
    subclass_rows.append({
        "child_uri": str(child),
        "child_class": readable_name(child),
        "parent_uri": str(parent),
        "parent_class": readable_name(parent)
    })

subclass_df = pd.DataFrame(subclass_rows)

print("Subclass axioms:", len(subclass_df))
display(subclass_df.head(100))


Subclass axioms: 42


,child_uri,child_class,parent_uri,parent_class
0,http://www.semanticweb.org/laptop-hp/ontologie...,Aerobic Exercise,http://www.semanticweb.org/laptop-hp/ontologie...,Exercise Activity
1,http://www.semanticweb.org/laptop-hp/ontologie...,Resistance Exercise,http://www.semanticweb.org/laptop-hp/ontologie...,Exercise Activity
2,http://www.semanticweb.org/laptop-hp/ontologie...,Borderline Hb A1c,http://www.semanticweb.org/laptop-hp/ontologie...,Hb A1c Level
3,http://www.semanticweb.org/laptop-hp/ontologie...,Diabetic Hb A1c,http://www.semanticweb.org/laptop-hp/ontologie...,Hb A1c Level
4,http://www.semanticweb.org/laptop-hp/ontologie...,Normal Hb A1c,http://www.semanticweb.org/laptop-hp/ontologie...,Hb A1c Level
5,http://www.semanticweb.org/laptop-hp/ontologie...,Calorie Restricted Diet,http://www.semanticweb.org/laptop-hp/ontologie...,Dietary Pattern
6,http://www.semanticweb.org/laptop-hp/ontologie...,High Sugar Diet,http://www.semanticweb.org/laptop-hp/ontologie...,Dietary Pattern
7,http://www.semanticweb.org/laptop-hp/ontologie...,Low Glycemic Index Diet,http://www.semanticweb.org/laptop-hp/ontologie...,Dietary Pattern
8,http://www.semanticweb.org/laptop-hp/ontologie...,Mediterranean Diet Pattern,http://www.semanticweb.org/laptop-hp/ontologie...,Dietary Pattern
9,http://www.semanticweb.org/laptop-hp/ontologie...,Cardiometabolic Risk Status,http://www.semanticweb.org/laptop-hp/ontologie...,Clinical Concept


In [168]:
object_property_uris = set(
    g.subjects(RDF.type, OWL.ObjectProperty)
)

object_property_rows = []

for property_uri in object_property_uris:
    domain_values = [
        readable_name(value)
        for value in g.objects(property_uri, RDFS.domain)
    ]
    
    range_values = [
        readable_name(value)
        for value in g.objects(property_uri, RDFS.range)
    ]
    
    assertion_count = sum(
        1 for _ in g.triples((None, property_uri, None))
    )
    
    object_property_rows.append({
        "property_uri": str(property_uri),
        "property_name": readable_name(property_uri),
        "domain": " | ".join(domain_values),
        "range": " | ".join(range_values),
        "assertion_count": assertion_count
    })

object_properties_df = pd.DataFrame(object_property_rows)

if not object_properties_df.empty:
    object_properties_df = object_properties_df.sort_values(
        "assertion_count",
        ascending=False
    ).reset_index(drop=True)

print("Object properties:", len(object_properties_df))
display(object_properties_df)


Object properties: 21


,property_uri,property_name,domain,range,assertion_count
0,http://www.semanticweb.org/laptop-hp/ontologie...,leads To,,,100030
1,http://www.semanticweb.org/laptop-hp/ontologie...,has BMI,Patient,BMI | float,768
2,http://www.semanticweb.org/laptop-hp/ontologie...,has Blood Pressure,Patient,Blood Pressure,768
3,http://www.semanticweb.org/laptop-hp/ontologie...,has Skin Thickness,Patient,Skin Thickness,768
4,http://www.semanticweb.org/laptop-hp/ontologie...,has Insulin,Patient,Insulin,768
5,http://www.semanticweb.org/laptop-hp/ontologie...,has Outcome,Patient,Outcome,768
6,http://www.semanticweb.org/laptop-hp/ontologie...,has Diabetes Pedigree Function,Patient,Diabetes Pedigree Function,768
7,http://www.semanticweb.org/laptop-hp/ontologie...,has Age,Patient,Age | byte,768
8,http://www.semanticweb.org/laptop-hp/ontologie...,has Pregnancies,Patient,Pregnancies,768
9,http://www.semanticweb.org/laptop-hp/ontologie...,has ID,Patient,ID,768


In [169]:
data_property_uris = set(
    g.subjects(RDF.type, OWL.DatatypeProperty)
)

data_property_rows = []

for property_uri in data_property_uris:
    domain_values = [
        readable_name(value)
        for value in g.objects(property_uri, RDFS.domain)
    ]
    
    range_values = [
        readable_name(value)
        for value in g.objects(property_uri, RDFS.range)
    ]
    
    assertion_count = sum(
        1 for _ in g.triples((None, property_uri, None))
    )
    
    data_property_rows.append({
        "property_uri": str(property_uri),
        "property_name": readable_name(property_uri),
        "domain": " | ".join(domain_values),
        "range": " | ".join(range_values),
        "assertion_count": assertion_count
    })

data_properties_df = pd.DataFrame(data_property_rows)

if not data_properties_df.empty:
    data_properties_df = data_properties_df.sort_values(
        "assertion_count",
        ascending=False
    ).reset_index(drop=True)

print("Datatype properties:", len(data_properties_df))
display(data_properties_df)


Datatype properties: 5


,property_uri,property_name,domain,range,assertion_count
0,http://www.semanticweb.org/laptop-hp/ontologie...,has BMI,Patient,BMI | float,768
1,http://www.semanticweb.org/laptop-hp/ontologie...,has Age,Patient,Age | byte,768
2,http://www.semanticweb.org/laptop-hp/ontologie...,has Glucose Level,Patient,float,0
3,http://www.semanticweb.org/laptop-hp/ontologie...,has Confidence Score,,,0
4,http://www.semanticweb.org/laptop-hp/ontologie...,has Hb A1c Value,Patient,float,0


In [170]:
patient_class_uris = {
    class_uri
    for class_uri in g.subjects(RDFS.subClassOf, None)
    if readable_name(class_uri).lower() == "patient"
}

patient_class_uris.update({
    class_uri
    for class_uri in g.objects(None, RDF.type)
    if readable_name(class_uri).lower() == "patient"
})

patient_class_uris = {
    class_uri for class_uri in patient_class_uris
    if isinstance(class_uri, URIRef)
}

patient_individuals = set()

for patient_class_uri in patient_class_uris:
    patient_individuals.update(
        g.subjects(RDF.type, patient_class_uri)
    )

direct_patient_individuals = set(
    g.subjects(RDF.type, next(iter(patient_class_uris)))
) if patient_class_uris else set()

print("Patient-related classes:", len(patient_class_uris))
print("Patient individuals:", len(patient_individuals))
print("Sample patients:", list(patient_individuals)[:20])


Patient-related classes: 1
Patient individuals: 768
Sample patients: [rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#177'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#534'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#607'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#463'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#547'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#103'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#693'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#539'), rdflib.term.URIRef('http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#1

In [171]:
def get_subclasses(graph, parent_class):
    subclasses = set()
    pending = [parent_class]
    
    while pending:
        current = pending.pop()
        
        for child in graph.subjects(RDFS.subClassOf, current):
            if child not in subclasses:
                subclasses.add(child)
                pending.append(child)
    
    return subclasses

patient_root = None

for class_uri in set(g.subjects(RDF.type, OWL.Class)):
    if readable_name(class_uri).lower() == "patient":
        patient_root = class_uri
        break

patient_related_classes = set()

if patient_root is not None:
    patient_related_classes.add(patient_root)
    patient_related_classes.update(get_subclasses(g, patient_root))

patient_individuals = set()

for class_uri in patient_related_classes:
    patient_individuals.update(g.subjects(RDF.type, class_uri))

print("Patient root:", patient_root)
print("Patient-related classes:", len(patient_related_classes))
print("Patient individuals:", len(patient_individuals))


Patient root: http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#Patient
Patient-related classes: 6
Patient individuals: 768


In [172]:
patient_assertion_rows = []

for patient in patient_individuals:
    patient_types = [
        readable_name(class_uri)
        for class_uri in g.objects(patient, RDF.type)
    ]
    
    for predicate, obj in g.predicate_objects(patient):
        if predicate == RDF.type:
            continue
        
        if predicate in {
            RDFS.label,
            RDFS.comment
        }:
            continue
        
        if isinstance(obj, Literal):
            object_type = "Literal"
            object_value = str(obj)
        elif isinstance(obj, URIRef):
            object_type = "URIRef"
            object_value = readable_name(obj)
        else:
            object_type = "BNode"
            object_value = str(obj)
        
        patient_assertion_rows.append({
            "patient_uri": str(patient),
            "patient_id": readable_name(patient),
            "patient_types": " | ".join(patient_types),
            "predicate_uri": str(predicate),
            "predicate": readable_name(predicate),
            "object_type": object_type,
            "object_uri": str(obj) if isinstance(obj, URIRef) else None,
            "object": object_value
        })

patient_assertions_df = pd.DataFrame(patient_assertion_rows)

print("Patient property assertions:", len(patient_assertions_df))
display(patient_assertions_df.head(10))


Patient property assertions: 7680


,patient_uri,patient_id,patient_types,predicate_uri,predicate,object_type,object_uri,object
0,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Age,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,42
1,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has BMI,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,31.2
2,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Blood Pressure,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,78
3,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Diabetes Pedigree Function,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,0.382
4,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Glucose,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,85
5,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has ID,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,177
6,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Insulin,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,0
7,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Outcome,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,0
8,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Pregnancies,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,6
9,http://www.semanticweb.org/laptop-hp/ontologie...,177,Named Individual | Patient,http://www.semanticweb.org/laptop-hp/ontologie...,has Skin Thickness,URIRef,http://www.semanticweb.org/laptop-hp/ontologie...,0


In [173]:
patient_474 = None

for subject in set(g.subjects()):
    if readable_name(subject) == "474":
        patient_474 = subject
        break

if patient_474 is None:
    print("Patient 474 was not found.")
else:
    print("Patient:", patient_474)
    
    for predicate, obj in g.predicate_objects(patient_474):
        print(
            readable_name(predicate),
            "->",
            readable_name(obj) if isinstance(obj, URIRef) else str(obj)
        )


Patient: http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#474
type -> Named Individual
type -> Patient
has Age -> 50
has BMI -> 29.9
has Blood Pressure -> 90
has Diabetes Pedigree Function -> 0.21
has Glucose -> 136
has ID -> 474
has Insulin -> 0
has Outcome -> 0
has Pregnancies -> 7
has Skin Thickness -> 0


In [174]:
patient_table_rows = []

for patient in patient_individuals:
    row = {
        "patient": readable_name(patient)
    }
    
    for predicate, obj in g.predicate_objects(patient):
        if predicate == RDF.type:
            continue
        
        property_name = readable_name(predicate)
        
        if isinstance(obj, Literal):
            value = str(obj)
        elif isinstance(obj, URIRef):
            value = readable_name(obj)
        else:
            value = str(obj)
        
        row[property_name] = value
    
    patient_table_rows.append(row)

patients_df = pd.DataFrame(patient_table_rows)

if not patients_df.empty:
    patients_df = patients_df.sort_values(
        "patient",
        key=lambda column: column.astype(str)
    ).reset_index(drop=True)

print("Patient rows:", len(patients_df))
display(patients_df.head(20))


Patient rows: 768


,patient,has Age,has BMI,has Blood Pressure,has Diabetes Pedigree Function,has Glucose,has ID,has Insulin,has Outcome,has Pregnancies,has Skin Thickness
0,1,50,33.6,72,0.627,148,1,0,1,6,35
1,10,54,0,96,0.232,125,10,0,1,8,0
2,100,31,49.7,90,0.325,122,100,220,1,1,51
3,101,33,39,72,1.222,163,101,0,1,1,0
4,102,22,26.1,60,0.179,151,102,0,0,1,0
5,103,21,22.5,96,0.262,125,103,0,0,0,0
6,104,24,26.6,72,0.283,81,104,40,0,1,18
7,105,27,39.6,65,0.93,85,105,0,0,2,0
8,106,21,28.7,56,0.801,126,106,152,0,1,29
9,107,27,22.4,122,0.207,96,107,0,0,1,0


In [175]:
def first_existing_value(row, names):
    for name in names:
        if name in row.index and pd.notna(row[name]):
            return row[name]
    return None

def build_patient_sentence(row):
    patient_id = row["patient"]
    parts = []
    
    age = first_existing_value(row, ["hasAge", "Age", "age"])
    bmi = first_existing_value(row, ["hasBMI", "BMI", "bmi"])
    glucose = first_existing_value(row, ["hasGlucose", "Glucose", "glucose"])
    blood_pressure = first_existing_value(
        row,
        ["hasBloodPressure", "BloodPressure", "blood pressure"]
    )
    insulin = first_existing_value(row, ["hasInsulin", "Insulin", "insulin"])
    outcome = first_existing_value(row, ["hasOutcome", "Outcome", "outcome"])
    
    if age is not None:
        parts.append(f"is {age} years old")
    
    if bmi is not None:
        parts.append(f"has a BMI of {bmi}")
    
    if glucose is not None:
        parts.append(f"has a glucose value of {glucose}")
    
    if blood_pressure is not None:
        parts.append(f"has a blood pressure value of {blood_pressure}")
    
    if insulin is not None:
        parts.append(f"has an insulin value of {insulin}")
    
    if outcome is not None:
        parts.append(f"has a diabetes outcome value of {outcome}")
    
    if not parts:
        return None
    
    return f"Patient {patient_id} " + ", ".join(parts) + "."

patients_df["sentence"] = patients_df.apply(build_patient_sentence, axis=1)

patient_sentences_df = patients_df[
    ["patient", "sentence"]
].dropna().drop_duplicates()

display(patient_sentences_df.head(20))


,patient,sentence


# Generate sentence Good enough for BERT

In [176]:
excluded_predicates = {
    "leads To",
    "causes"
}


In [177]:
from pathlib import Path
from collections import defaultdict
from rdflib import URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD
import pandas as pd
import re
import random

random.seed(42)

output_dir = Path("/kaggle/working/diabetes_ontology_mlm")
output_dir.mkdir(parents=True, exist_ok=True)

def local_name(value):
    text = str(value)
    
    if "#" in text:
        text = text.rsplit("#", 1)[-1]
    
    text = text.rstrip("/").rsplit("/", 1)[-1]
    return text

def readable_name(value):
    name = local_name(value)
    name = name.replace("_", " ")
    name = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", name)
    name = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

def normalize_text(text):
    text = str(text)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_sentence(text):
    text = normalize_text(text)
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" ,", ",")
    text = text.replace(" .", ".")
    text = text.strip()
    
    if text and not text.endswith("."):
        text += "."
    
    return text

def value_to_text(value):
    if isinstance(value, Literal):
        return str(value)
    
    if isinstance(value, URIRef):
        return readable_name(value)
    
    return str(value)

def numeric_value(value):
    try:
        return float(value_to_text(value))
    except (TypeError, ValueError):
        return None


In [178]:
def is_ontology_class(value):
    if not isinstance(value, URIRef):
        return False
    
    text = str(value)
    
    excluded_namespaces = {
        str(RDF),
        str(RDFS),
        str(OWL),
        str(XSD)
    }
    
    if any(text.startswith(namespace) for namespace in excluded_namespaces):
        return False
    
    excluded_uris = {
        RDF.Property,
        OWL.Class,
        OWL.ObjectProperty,
        OWL.DatatypeProperty,
        OWL.AnnotationProperty,
        OWL.NamedIndividual,
        OWL.Restriction
    }
    
    return value not in excluded_uris

class_uris = set()

for class_uri in g.subjects(RDF.type, OWL.Class):
    if is_ontology_class(class_uri):
        class_uris.add(class_uri)

for class_uri in g.subjects(RDF.type, RDFS.Class):
    if is_ontology_class(class_uri):
        class_uris.add(class_uri)

for child_uri, parent_uri in g.subject_objects(RDFS.subClassOf):
    if is_ontology_class(child_uri):
        class_uris.add(child_uri)
    
    if is_ontology_class(parent_uri):
        class_uris.add(parent_uri)

for _, class_uri in g.subject_objects(RDF.type):
    if is_ontology_class(class_uri):
        class_uris.add(class_uri)

class_rows = []

for class_uri in sorted(class_uris, key=lambda value: readable_name(value)):
    parent_classes = [
        readable_name(parent_uri)
        for parent_uri in g.objects(class_uri, RDFS.subClassOf)
        if is_ontology_class(parent_uri)
    ]
    
    instance_count = sum(
        1 for _ in g.subjects(RDF.type, class_uri)
    )
    
    direct_subclass_count = sum(
        1 for _ in g.subjects(RDFS.subClassOf, class_uri)
    )
    
    class_rows.append({
        "class_uri": str(class_uri),
        "class_name": readable_name(class_uri),
        "instance_count": instance_count,
        "direct_subclass_count": direct_subclass_count,
        "parent_classes": " | ".join(parent_classes)
    })

classes_df = pd.DataFrame(class_rows)

classes_df = classes_df.sort_values(
    "class_name"
).reset_index(drop=True)

print("Real ontology classes:", len(classes_df))
display(classes_df)


Real ontology classes: 65


,class_uri,class_name,instance_count,direct_subclass_count,parent_classes
0,http://www.semanticweb.org/laptop-hp/ontologie...,Aerobic Exercise,1,1,Exercise Activity
1,http://www.semanticweb.org/laptop-hp/ontologie...,Age,0,0,
2,http://www.semanticweb.org/laptop-hp/ontologie...,BMI,0,4,
3,http://www.semanticweb.org/laptop-hp/ontologie...,Blood Pressure,0,4,
4,http://www.semanticweb.org/laptop-hp/ontologie...,Borderline Hb A1c,0,0,Hb A1c Level
...,...,...,...,...,...
60,http://www.semanticweb.org/laptop-hp/ontologie...,Target Glycemic Control,1,0,Glycemic Control Status
61,http://www.semanticweb.org/laptop-hp/ontologie...,Underweight,0,0,BMI
62,http://www.semanticweb.org/laptop-hp/ontologie...,Walking Exercise,1,0,Aerobic Exercise
63,http://www.semanticweb.org/laptop-hp/ontologie...,Weight Loss Intervention,1,0,Lifestyle Factor


In [179]:
patient_class = None

for class_uri in class_uris:
    if readable_name(class_uri).lower() == "patient":
        patient_class = class_uri
        break

if patient_class is None:
    raise ValueError("The Patient class was not found.")

patient_individuals = set(
    g.subjects(RDF.type, patient_class)
)

print("Patient class:", patient_class)
print("Patient individuals:", len(patient_individuals))


Patient class: http://www.semanticweb.org/laptop-hp/ontologies/2024/10/untitled-ontology-13#Patient
Patient individuals: 768


In [180]:
patient_records = {}

for patient_uri in patient_individuals:
    record = {
        "uri": patient_uri,
        "patient_id": readable_name(patient_uri),
        "types": [],
        "properties": {}
    }
    
    for class_uri in g.objects(patient_uri, RDF.type):
        class_name = readable_name(class_uri)
        
        if class_name.lower() not in {
            "named individual",
            "patient"
        }:
            record["types"].append(class_name)
    
    for predicate_uri, object_value in g.predicate_objects(patient_uri):
        if predicate_uri == RDF.type:
            continue
        
        predicate_name = readable_name(predicate_uri)
        record["properties"][predicate_name] = value_to_text(object_value)
    
    patient_records[patient_uri] = record

print("Prepared patient records:", len(patient_records))


Prepared patient records: 768


In [181]:
sample_patient = next(iter(patient_records.values()))

print(sample_patient["patient_id"])
print(sample_patient["types"])
print(sample_patient["properties"])


177
[]
{'has Age': '42', 'has BMI': '31.2', 'has Blood Pressure': '78', 'has Diabetes Pedigree Function': '0.382', 'has Glucose': '85', 'has ID': '177', 'has Insulin': '0', 'has Outcome': '0', 'has Pregnancies': '6', 'has Skin Thickness': '0'}


In [182]:
def get_patient_number(record, property_names):
    for property_name in property_names:
        if property_name in record["properties"]:
            value = numeric_value(record["properties"][property_name])
            
            if value is not None:
                return value
    
    return None

def get_patient_value(record, property_names):
    for property_name in property_names:
        if property_name in record["properties"]:
            return record["properties"][property_name]
    
    return None


In [183]:
def generate_profile_sentence(record):
    patient_id = record["patient_id"]
    parts = []
    
    age = get_patient_number(record, ["has Age"])
    bmi = get_patient_number(record, ["has BMI"])
    glucose = get_patient_number(record, ["has Glucose"])
    blood_pressure = get_patient_number(record, ["has Blood Pressure"])
    insulin = get_patient_number(record, ["has Insulin"])
    pregnancies = get_patient_number(record, ["has Pregnancies"])
    skin_thickness = get_patient_number(record, ["has Skin Thickness"])
    pedigree = get_patient_number(
        record,
        ["has Diabetes Pedigree Function"]
    )
    outcome = get_patient_number(record, ["has Outcome"])
    
    if age is not None:
        parts.append(f"is {age:g} years old")
    
    if pregnancies is not None:
        parts.append(f"has {pregnancies:g} pregnancies")
    
    if glucose is not None:
        parts.append(f"has a glucose value of {glucose:g}")
    
    if blood_pressure is not None:
        parts.append(f"has a blood pressure value of {blood_pressure:g}")
    
    if skin_thickness is not None:
        parts.append(f"has a skin thickness value of {skin_thickness:g}")
    
    if insulin is not None:
        parts.append(f"has an insulin value of {insulin:g}")
    
    if bmi is not None:
        parts.append(f"has a BMI of {bmi:g}")
    
    if pedigree is not None:
        parts.append(
            f"has a diabetes pedigree function value of {pedigree:g}"
        )
    
    if outcome is not None:
        outcome_text = "positive" if outcome == 1 else "negative"
        parts.append(
            f"has a {outcome_text} diabetes outcome"
        )
    
    if not parts:
        return None
    
    return clean_sentence(
        f"Patient {patient_id} " + ", ".join(parts)
    )


In [184]:
def generate_measurement_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    age = get_patient_number(record, ["has Age"])
    bmi = get_patient_number(record, ["has BMI"])
    glucose = get_patient_number(record, ["has Glucose"])
    blood_pressure = get_patient_number(record, ["has Blood Pressure"])
    insulin = get_patient_number(record, ["has Insulin"])
    pregnancies = get_patient_number(record, ["has Pregnancies"])
    skin_thickness = get_patient_number(record, ["has Skin Thickness"])
    pedigree = get_patient_number(
        record,
        ["has Diabetes Pedigree Function"]
    )
    outcome = get_patient_number(record, ["has Outcome"])
    
    if age is not None:
        sentences.extend([
            f"Patient {patient_id} is {age:g} years old.",
            f"The age of patient {patient_id} is {age:g} years."
        ])
    
    if bmi is not None:
        sentences.extend([
            f"Patient {patient_id} has a BMI of {bmi:g}.",
            f"The BMI measurement for patient {patient_id} is {bmi:g}."
        ])
    
    if glucose is not None:
        sentences.extend([
            f"Patient {patient_id} has a glucose value of {glucose:g}.",
            f"The glucose measurement for patient {patient_id} is {glucose:g}."
        ])
    
    if blood_pressure is not None:
        sentences.extend([
            f"Patient {patient_id} has a blood pressure value of {blood_pressure:g}.",
            f"The blood pressure measurement for patient {patient_id} is {blood_pressure:g}."
        ])
    
    if insulin is not None:
        sentences.extend([
            f"Patient {patient_id} has an insulin value of {insulin:g}.",
            f"The insulin measurement for patient {patient_id} is {insulin:g}."
        ])
    
    if pregnancies is not None:
        sentences.extend([
            f"Patient {patient_id} has experienced {pregnancies:g} pregnancies.",
            f"The number of pregnancies for patient {patient_id} is {pregnancies:g}."
        ])
    
    if skin_thickness is not None:
        sentences.extend([
            f"Patient {patient_id} has a skin thickness value of {skin_thickness:g}.",
            f"The skin thickness measurement for patient {patient_id} is {skin_thickness:g}."
        ])
    
    if pedigree is not None:
        sentences.extend([
            f"Patient {patient_id} has a diabetes pedigree function value of {pedigree:g}.",
            f"The diabetes pedigree function for patient {patient_id} is {pedigree:g}."
        ])
    
    if outcome is not None:
        outcome_text = "positive" if outcome == 1 else "negative"
        
        sentences.extend([
            f"Patient {patient_id} has a {outcome_text} diabetes outcome.",
            f"The diabetes outcome for patient {patient_id} is {outcome_text}."
        ])
    
    return [clean_sentence(sentence) for sentence in sentences]


In [185]:
def generate_clinical_interpretation_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    bmi = get_patient_number(record, ["has BMI"])
    glucose = get_patient_number(record, ["has Glucose"])
    age = get_patient_number(record, ["has Age"])
    outcome = get_patient_number(record, ["has Outcome"])
    pregnancies = get_patient_number(record, ["has Pregnancies"])
    
    if bmi is not None:
        if bmi < 18.5:
            sentences.append(
                f"Patient {patient_id} has a BMI in the underweight range."
            )
        elif bmi < 25:
            sentences.append(
                f"Patient {patient_id} has a BMI in the normal weight range."
            )
        elif bmi < 30:
            sentences.append(
                f"Patient {patient_id} has a BMI in the overweight range."
            )
        else:
            sentences.append(
                f"Patient {patient_id} has a BMI in the obese range."
            )
    
    if glucose is not None:
        if glucose < 100:
            sentences.append(
                f"Patient {patient_id} has a glucose value below the diabetes screening threshold."
            )
        elif glucose < 126:
            sentences.append(
                f"Patient {patient_id} has a glucose value in the elevated range."
            )
        else:
            sentences.append(
                f"Patient {patient_id} has a glucose value in the diabetes range."
            )
    
    if age is not None and age >= 65:
        sentences.append(
            f"Patient {patient_id} is in an older age group."
        )
    
    if pregnancies is not None and pregnancies > 0:
        sentences.append(
            f"Patient {patient_id} has a history of pregnancy."
        )
    
    if outcome is not None:
        if outcome == 1:
            sentences.append(
                f"Patient {patient_id} is associated with a positive diabetes outcome."
            )
        else:
            sentences.append(
                f"Patient {patient_id} is associated with a negative diabetes outcome."
            )
    
    return [clean_sentence(sentence) for sentence in sentences]


In [186]:
def generate_type_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    for type_name in sorted(set(record["types"])):
        type_lower = type_name.lower()
        
        if "diabetic" in type_lower:
            sentence = (
                f"Patient {patient_id} is classified as a diabetic patient."
            )
        elif "non diabetic" in type_lower:
            sentence = (
                f"Patient {patient_id} is classified as a non diabetic patient."
            )
        elif "elderly" in type_lower:
            sentence = (
                f"Patient {patient_id} is classified as an elderly patient."
            )
        elif "high risk" in type_lower:
            sentence = (
                f"Patient {patient_id} is classified as a high risk patient."
            )
        elif "pregnant" in type_lower:
            sentence = (
                f"Patient {patient_id} is classified as a pregnant patient."
            )
        else:
            sentence = (
                f"Patient {patient_id} belongs to the {type_name} category."
            )
        
        sentences.append(clean_sentence(sentence))
    
    return sentences


In [187]:
def generate_combined_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    age = get_patient_number(record, ["has Age"])
    bmi = get_patient_number(record, ["has BMI"])
    glucose = get_patient_number(record, ["has Glucose"])
    outcome = get_patient_number(record, ["has Outcome"])
    
    if age is not None and bmi is not None:
        sentences.append(
            f"Patient {patient_id} is {age:g} years old and has a BMI of {bmi:g}."
        )
    
    if glucose is not None and outcome is not None:
        outcome_text = "positive" if outcome == 1 else "negative"
        sentences.append(
            f"Patient {patient_id} has a glucose value of {glucose:g} and a {outcome_text} diabetes outcome."
        )
    
    if bmi is not None and glucose is not None:
        sentences.append(
            f"Patient {patient_id} has a BMI of {bmi:g} and a glucose value of {glucose:g}."
        )
    
    if age is not None and glucose is not None and outcome is not None:
        outcome_text = "positive" if outcome == 1 else "negative"
        sentences.append(
            f"Patient {patient_id} is {age:g} years old, has a glucose value of {glucose:g}, and has a {outcome_text} diabetes outcome."
        )
    
    return [clean_sentence(sentence) for sentence in sentences]


In [188]:
def generate_class_hierarchy_sentences(graph):
    sentences = []
    
    for child_uri, parent_uri in graph.subject_objects(RDFS.subClassOf):
        if not is_ontology_class(child_uri):
            continue
        
        if not is_ontology_class(parent_uri):
            continue
        
        child_name = readable_name(child_uri)
        parent_name = readable_name(parent_uri)
        
        templates = [
            f"{child_name} is a type of {parent_name}.",
            f"The {child_name} class is a subclass of {parent_name}.",
            f"An entity classified as {child_name} belongs to the broader {parent_name} category."
        ]
        
        sentences.extend(templates)
    
    return [clean_sentence(sentence) for sentence in sentences]


In [189]:
def generate_property_schema_sentences(graph):
    sentences = []
    
    property_uris = set(
        graph.subjects(RDF.type, OWL.ObjectProperty)
    )
    
    property_uris.update(
        graph.subjects(RDF.type, OWL.DatatypeProperty)
    )
    
    for property_uri in property_uris:
        if not isinstance(property_uri, URIRef):
            continue
        
        property_name = readable_name(property_uri)
        
        domains = [
            readable_name(domain_uri)
            for domain_uri in graph.objects(property_uri, RDFS.domain)
            if is_ontology_class(domain_uri)
        ]
        
        ranges = [
            readable_name(range_uri)
            for range_uri in graph.objects(property_uri, RDFS.range)
            if is_ontology_class(range_uri)
        ]
        
        for domain_name in domains:
            sentences.append(
                f"The {property_name} property describes a relationship from {domain_name}."
            )
            
            for range_name in ranges:
                sentences.append(
                    f"The {property_name} property connects a {domain_name} to a {range_name}."
                )
    
    return [clean_sentence(sentence) for sentence in sentences]


In [190]:
def generate_patient_property_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    for property_name, property_value in record["properties"].items():
        if property_name in excluded_predicates:
            continue
        
        if property_name in {
            "has ID",
            "has Age",
            "has BMI",
            "has Blood Pressure",
            "has Diabetes Pedigree Function",
            "has Glucose",
            "has Insulin",
            "has Outcome",
            "has Pregnancies",
            "has Skin Thickness"
        }:
            continue
        
        value_text = value_to_text(property_value)
        
        sentences.extend([
            f"Patient {patient_id} has a {property_name} value of {value_text}.",
            f"The {property_name} associated with patient {patient_id} is {value_text}."
        ])
    
    return [clean_sentence(sentence) for sentence in sentences]


In [191]:
def generate_patient_property_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    for property_name, property_value in record["properties"].items():
        if property_name in excluded_predicates:
            continue
        
        if property_name in {
            "has ID",
            "has Age",
            "has BMI",
            "has Blood Pressure",
            "has Diabetes Pedigree Function",
            "has Glucose",
            "has Insulin",
            "has Outcome",
            "has Pregnancies",
            "has Skin Thickness"
        }:
            continue
        
        value_text = value_to_text(property_value)
        
        sentences.extend([
            f"Patient {patient_id} has a {property_name} value of {value_text}.",
            f"The {property_name} associated with patient {patient_id} is {value_text}."
        ])
    
    return [clean_sentence(sentence) for sentence in sentences]


In [192]:
def generate_clinical_interpretation_sentences(record):
    patient_id = record["patient_id"]
    sentences = []
    
    bmi = get_patient_number(record, ["has BMI"])
    glucose = get_patient_number(record, ["has Glucose"])
    age = get_patient_number(record, ["has Age"])
    outcome = get_patient_number(record, ["has Outcome"])
    pregnancies = get_patient_number(record, ["has Pregnancies"])
    
    if bmi is not None:
        if bmi < 18.5:
            sentences.append(
                f"Patient {patient_id} has a BMI in the underweight range."
            )
        elif bmi < 25:
            sentences.append(
                f"Patient {patient_id} has a BMI in the normal weight range."
            )
        elif bmi < 30:
            sentences.append(
                f"Patient {patient_id} has a BMI in the overweight range."
            )
        else:
            sentences.append(
                f"Patient {patient_id} has a BMI in the obese range."
            )
    
    if glucose is not None:
        if glucose < 100:
            sentences.append(
                f"Patient {patient_id} has a glucose value below the diabetes screening threshold."
            )
        elif glucose < 126:
            sentences.append(
                f"Patient {patient_id} has a glucose value in the elevated range."
            )
        else:
            sentences.append(
                f"Patient {patient_id} has a glucose value in the diabetes range."
            )
    
    if age is not None and age >= 65:
        sentences.append(
            f"Patient {patient_id} is in an older age group."
        )
    
    if pregnancies is not None and pregnancies > 0:
        sentences.append(
            f"Patient {patient_id} has a history of pregnancy."
        )
    
    if outcome is not None:
        if outcome == 1:
            sentences.append(
                f"Patient {patient_id} is associated with a positive diabetes outcome."
            )
        else:
            sentences.append(
                f"Patient {patient_id} is associated with a negative diabetes outcome."
            )
    
    return [clean_sentence(sentence) for sentence in sentences]


In [193]:
all_sentences = []

for record in patient_records.values():
    profile_sentence = generate_profile_sentence(record)
    
    if profile_sentence:
        all_sentences.append(profile_sentence)
    
    all_sentences.extend(
        generate_measurement_sentences(record)
    )
    
    all_sentences.extend(
        generate_clinical_interpretation_sentences(record)
    )
    
    all_sentences.extend(
        generate_type_sentences(record)
    )
    
    all_sentences.extend(
        generate_combined_sentences(record)
    )
    
    all_sentences.extend(
        generate_patient_property_sentences(record)
    )

all_sentences.extend(
    generate_class_hierarchy_sentences(g)
)

all_sentences.extend(
    generate_property_schema_sentences(g)
)

all_sentences = [
    clean_sentence(sentence)
    for sentence in all_sentences
    if sentence and len(sentence.split()) >= 5
]

all_sentences = list(dict.fromkeys(all_sentences))

random.shuffle(all_sentences)

corpus_path = output_dir / "final_ontology_corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as file:
    for sentence in all_sentences:
        file.write(sentence + "\n")

print("Final sentence count:", len(all_sentences))
print("Corpus path:", corpus_path)


Final sentence count: 20807
Corpus path: /kaggle/working/diabetes_ontology_mlm/final_ontology_corpus.txt
